In [16]:
import os
import re
import gc
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path

In [ ]:
# =====================================================
# 0. FEATURES CALCULATION FUNCTIONS
# =====================================================
def weighted_global_efficiency(G, weight='distance'):
    n = len(G)
    if n < 2:
        return 0.0

    total = 0.0
    for node in G.nodes():
        # Shortest path lengths using the distance attribute
        lengths = nx.single_source_dijkstra_path_length(G, node, weight=weight)
        total += sum(1.0 / d for target, d in lengths.items() if target != node and d > 0)

    return total / (n * (n - 1))

def compute_features(mat, ratio_thres):
    G = nx.from_numpy_array(mat)
    
    # Group 1: features extracted from the original connectivity matrix
    cluster_coeff = nx.clustering(G, weight='weight') 
    strengths = dict(G.degree(weight='weight'))
    
    # Eigenvector centrality
    try:
        eigen_central = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
    except Exception:
        eigen_central = {node: 0.0 for node in G.nodes()}
    D = np.where(mat > 0, 1/mat, 0)
    D = nx.from_numpy_array(D)
    if not nx.is_connected(D):
        path_length = np.nan
    else:
        path_length = nx.average_shortest_path_length(D, weight='weight')
    
    # Group 2: Thresholding matrix
    thres = ratio_thres * mat.max()
    mat_ = np.where(mat > thres, mat, 0)
    G_ = nx.from_numpy_array(mat_)
    between_central = nx.betweenness_centrality(G_, weight='weight', normalized=True)
    dens = nx.density(G_)
    wge = weighted_global_efficiency(G_, weight='distance')

    
    df_features = pd.DataFrame({
        'Node': list(cluster_coeff.keys()),
        'Clustering_Coeff': list(cluster_coeff.values()),
        'Strength': [strengths[node] for node in cluster_coeff.keys()],
        'Eigen_Centrality': [eigen_central[node] for node in cluster_coeff.keys()],
        'Betweenness_Centrality': [between_central[node] for node in cluster_coeff.keys()],
        'Density': dens,
        'Weighted_Global_Efficiency': wge,
        'Characteristic_Path_Length':path_length
    })
    return df_features

# Collect all features and organize them in a dataframe
def compute_all_features(pli_current, ratio_thres):
    
    df_current = compute_features(pli_current, ratio_thres)
    
    df_curr_indexed = df_current.set_index('Node')
    feature_cols = [col for col in df_current.columns if col != 'Node']
    
    df_result = pd.DataFrame(df_curr_indexed)
    
    series_flat = df_result.unstack()
    
    flat_dict = {}
    for (feat_name, node_id), val in series_flat.items():
        flat_dict[f"Node_{node_id}_{feat_name}"] = val
        
    return flat_dict

In [18]:
def extract_graph_features(folder_path,label, step=None,ratio_thres=0.7):
    
    folder = Path(folder_path)
    files = sorted([f for f in folder.glob('*.npy')])
    
    if not files:
        return pd.DataFrame()

    rows_list = []
    for file_path in files:
        
        band = file_path.name.split('.')[0]            
        print('Band: ', band)
        file_id = file_path.parent.name # 'seizure ID or batch index'
        
        matrices = np.load(file_path)
                
        for epoch_idx, matrix in enumerate(matrices):
            feat_dict = compute_all_features(matrix, ratio_thres)
            if label=='preictal':
                row_data = {
                    'SeizureID': file_id,
                    'Band': band,
                    'Epoch': epoch_idx
                }
            else:
                row_data = {
                    'Band': band,
                    'Epoch': epoch_idx+step
                }
                print('Epoch', epoch_idx+step)
            row_data.update(feat_dict)
            rows_list.append(row_data)

    final_df = pd.DataFrame(rows_list)
    return final_df, matrices.shape[0]

In [ ]:
# ==========================================================================
# MAIN EXECUTION
# ==========================================================================
mode = 'wPLI' # mode = 'PLI'
if __name__ == '__main__':
    base_path = Path(f'/kaggle/input/datasets/ngoclannguyen00/chb-mit-connectivity-training/{mode}_matrix')
    output_dir = Path(f'{mode}_Features_Nodal_Thres_0.5')

    for chb_folder in sorted(base_path.iterdir()):
        if not chb_folder.is_dir():
            continue
            
        num_chb = chb_folder.name  # 'chb01', 'chb02'...
        chb_output = output_dir / f'{num_chb}'
        chb_output.mkdir(parents=True, exist_ok=True)
        print(f'Processing {num_chb}...')
        
        for label_path in chb_folder.glob('*'):
            df_list = []
            label = label_path.name
            print('Label: ', label)
            prev_batch_size = 0
            step=0
            for epochs_path in sorted(label_path.glob('*')):
                if label=='preictal':
                    print('Seizure ID: ', epochs_path.name)
                batch_idx = epochs_path.name
                if label=='interictal':
                    step += prev_batch_size
                else:
                    step==None
                if epochs_path.is_dir():
                    features, prev_batch_size = extract_graph_features(epochs_path, label, step=step, ratio_thres=0.5)
                    if not features.empty:
                        df_list.append(features)
                
            if df_list:
                df_result = pd.concat(df_list, ignore_index=True)
                save_path = chb_output / f'{label}.csv'
                df_result.to_csv(save_path, index=False)
                print(f'  [OK] Completed {num_chb} -> Saved to {save_path}')
                
                del df_list, df_result
            gc.collect()

Processing chb01...
Label:  interictal
Band:  alpha
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7


/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642

Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14
Epoch 15
Epoch 16
Epoch 17
Epoch 18


/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642

Epoch 19
Epoch 20
Epoch 21
Epoch 22
Epoch 23
Epoch 24
Epoch 25
Epoch 26
Epoch 27
Epoch 28
Epoch 29
Epoch 30
Epoch 31


...

Epoch 92
Epoch 93
Epoch 94
Epoch 95
Epoch 96
Epoch 97
Epoch 98
Epoch 99
Epoch 100
Epoch 101
Epoch 102
Epoch 103


/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642

Epoch 104
Epoch 105
Epoch 106
Epoch 107
Epoch 108
Epoch 109
Epoch 110
Epoch 111
Epoch 112
Epoch 113
Epoch 114
Epoch 115


/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)
/tmp/ipykernel_58/296642832.py:30: RuntimeWarning: divide by zero encountered in divide
  D = np.where(mat > 0, 1/mat, 0)


KeyboardInterrupt: 